In [ ]:
%%configure -f
{"vCores": 4, "defaultLakehouse": {"name": "diagnostic", "id": "9d10bce5-1edc-4875-83c4-ac0a98a02775", "workspaceId": "82ad2591-974a-4ad4-ace6-e24879274a4b"}}

# fabric-rlm 0.1.11.dev2 — **PVR ablation** (Plan / Verify / Reflect)

Same RLM + bandit, **two conditions** per case:
* **OFF** — `FABRIC_RLM_PVR=0` strips PLAN/VERIFY/REFLECT clauses from `core.md`
  and disables synthesized REFLECT injection (legacy validator-feedback only).
* **ON**  — full PVR scaffold.

Each condition starts from **fresh in-memory bandit state** so the comparison
is not contaminated by warm-state ladder skipping.


In [ ]:
import sys, json, time, traceback, uuid, platform as _platform
from pathlib import Path

TIER = 'effort_bandit_diag'
RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]
FILES_ROOT = Path('/lakehouse/default/Files')
RUN_ROOT = FILES_ROOT / 'fabric_rlm_adaptive_validation' / TIER / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Separate state.json so this doesn't collide with the cross-model bandit priors.
BANDIT_STATE_PATH = FILES_ROOT / 'fabric_rlm_adaptive_validation' / 'effort_bandit_diag' / 'state.json'
BANDIT_STATE_PATH.parent.mkdir(parents=True, exist_ok=True)

summary = {
    'tier': TIER, 'run_id': RUN_ID, 'started_at': time.time(),
    'python': _platform.python_version(),
    'stages': [], 'iterations': [], 'passed': False, 'error': None,
}
SUMMARY_PATH = RUN_ROOT / 'summary.json'

def write_summary():
    summary['updated_at'] = time.time()
    summary['elapsed_seconds'] = summary['updated_at'] - summary['started_at']
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')

def stage(name, **fields):
    summary['stages'].append({'stage': name, 't': time.time(), **fields})
    print(f'[stage] {name}', fields if fields else '')
    write_summary()

stage('setup', run_root=str(RUN_ROOT), bandit_state=str(BANDIT_STATE_PATH))


In [ ]:
WHEEL_PATH = '/lakehouse/default/Files/fabric_rlm_longcot/wheels/fabric_rlm-0.1.11.dev2-py3-none-any.whl'
import os
RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]
TIER = 'pvr_ablation'
FILES_ROOT = pathlib.Path('/lakehouse/default/Files')
RUN_ROOT = FILES_ROOT / 'fabric_rlm_adaptive_validation' / TIER / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = RUN_ROOT / 'summary.json'
summary = {
    'tier': TIER, 'run_id': RUN_ID, 'started_at': time.time(),
    'host': platform.node(), 'python': platform.python_version(),
    'wheel': WHEEL_PATH, 'stages': [], 'iterations': [],
}
def stage(name, **info):
    rec = {'stage': name, 't': time.time() - summary['started_at'], **info}
    summary['stages'].append(rec)
    write_summary()
    print(name, info)
def write_summary():
    summary['elapsed_seconds'] = time.time() - summary['started_at']
    SUMMARY_PATH.write_text(_json.dumps(summary, default=str, indent=2))
import json as _json
stage('setup', run_id=RUN_ID)
import os, subprocess
if not os.path.exists(WHEEL_PATH):
    raise FileNotFoundError(WHEEL_PATH)
stage('wheel_check', exists=True, size=os.path.getsize(WHEEL_PATH))
subprocess.check_call(['pip','install','--quiet','--force-reinstall','--no-deps', WHEEL_PATH])
stage('pip_wheel', done=True)
subprocess.check_call(['pip','install','--quiet','dspy>=3.0.4'])
stage('pip_dspy', done=True)
import dspy, fabric_rlm
stage('imported', dspy=dspy.__version__, fabric_rlm=fabric_rlm.__version__)


In [ ]:
import json as _json
FIXTURE_ROOT = '/lakehouse/default/Files/fabric_rlm_adaptive_validation/fixtures'

CASES = []
# 2 easy cases (math, csv) — fast-path generalization
with open(FIXTURE_ROOT + '/easy_cases.jsonl') as fh:
    easy = [_json.loads(line) for line in fh]
for cid in ['easy-math-1', 'easy-csv-1']:
    r = next(c for c in easy if c['id'] == cid)
    CASES.append({'id': r['id'], 'question': r['question'],
                  'answer': r['answer'], 'template': r.get('template') or 'easy',
                  'difficulty': 'easy'})

# 2 hard cases (the originals: PVR helped on Backprop, ceiling on VLIW)
with open(FIXTURE_ROOT + '/longcot_cs_hard_pilot20.jsonl') as fh:
    hard_rows = [_json.loads(line) for line in fh]
for tmpl in ['Backprop', 'VLIW']:
    matches = [r for r in hard_rows if r.get('template') == tmpl]
    if matches:
        r = matches[0]
        CASES.append({'id': tmpl.lower() + '-' + str(r['question_id']),
                      'question': r['prompt'], 'answer': r['answer'],
                      'template': tmpl, 'difficulty': 'hard'})

stage('cases_loaded', n=len(CASES), ids=[c['id'] for c in CASES])


In [ ]:
import sys
sys.path.insert(0, '/lakehouse/default/Files/fabric_rlm_adaptive_validation/fixtures')
try:
    from longcot_adapter import verify_cs_response
    HAS_LONGCOT = True
except Exception as _e:
    HAS_LONGCOT = False
    print('longcot_adapter import failed:', _e)

def normalize(s):
    return ''.join((s or '').lower().split())

def make_validator(case):
    expected_answer = case.get('answer')
    if expected_answer is None:
        expected_answer = ''
    elif not isinstance(expected_answer, str):
        try:
            expected_answer = _json.dumps(expected_answer, sort_keys=True)
        except Exception:
            expected_answer = str(expected_answer)
    template = case.get('template')
    if case.get('difficulty') == 'hard' and template and HAS_LONGCOT:
        def validator(result):
            if not result.submitted or not result.payload:
                return False
            ans = result.payload.get('answer') or ''
            try:
                correct, _ = verify_cs_response(template, expected_answer, ans)
                return bool(correct)
            except Exception:
                return normalize(expected_answer) in normalize(ans)
        return validator
    norm_expected = normalize(expected_answer)
    def validator(result):
        if not result.submitted or not result.payload:
            return False
        ans = result.payload.get('answer') or ''
        return norm_expected in normalize(ans)
    return validator


In [ ]:
from fabric_rlm import RLM, FabricLM
from fabric_rlm.experimental import BanditState, EffortBanditPolicy
import os

base_lm = FabricLM('gpt-5', reasoning_effort='minimal', cache=False)
stage('lm_built', base='gpt-5', start_effort='minimal')

CONDITIONS = [
    ('off', '0'),  # PVR disabled — baseline
    ('on',  '1'),  # PVR enabled — treatment
]

ablation = {'cases': []}
summary['ablation'] = ablation
write_summary()

for case in CASES:
    case_record = {'id': case['id'], 'template': case['template'],
                   'difficulty': case['difficulty'], 'conditions': {}}
    ablation['cases'].append(case_record)
    write_summary()
    for cond_name, env_val in CONDITIONS:
        os.environ['FABRIC_RLM_PVR'] = env_val
        # FRESH bandit state per (case, condition) — no cross-contamination.
        state = BanditState()
        try:
            validator = make_validator(case)
            policy = EffortBanditPolicy(
                state=state,
                task_key=case['template'],
                warmup=2,
                base_lm_spec=base_lm,
                base_reasoning_effort='minimal',
                parallel_rollouts=3,
            )
            rlm = RLM(
                signature='question -> answer',
                lm=base_lm,
                engine='adaptive',
                adaptive=dict(
                    policy=policy,
                    validator=validator,
                    max_attempts=6,
                    parallel_rollouts=1,
                ),
            )
            t0 = time.perf_counter()
            result = rlm.run({'question': case['question']})
            elapsed = time.perf_counter() - t0
            meta = (result.trajectory.metadata or {}).get('adaptive', {}) if result.trajectory else {}
            attempts = meta.get('attempts', [])
            passed_now = bool(result.submitted and validator(result))
            tot_prompt = sum((a.get('prompt_tokens') or 0) for a in attempts)
            tot_compl = sum((a.get('completion_tokens') or 0) for a in attempts)
            cond_record = {
                'passed': passed_now, 'submitted': result.submitted,
                'elapsed_seconds': elapsed,
                'starting_rung': attempts[0].get('rung') if attempts else None,
                'winner_rung': meta.get('winner_rung'),
                'stop_reason': meta.get('stop_reason'),
                'n_attempts': len(attempts),
                'total_prompt_tokens': tot_prompt,
                'total_completion_tokens': tot_compl,
                'attempts': attempts,
            }
        except Exception as exc:
            cond_record = {'passed': False, 'error': repr(exc),
                           'traceback': traceback.format_exc()}
        case_record['conditions'][cond_name] = cond_record
        write_summary()
        stage('cond_done', case=case['id'], cond=cond_name,
              passed=cond_record.get('passed'),
              n_attempts=cond_record.get('n_attempts'),
              elapsed=round(cond_record.get('elapsed_seconds') or 0, 1),
              tokens=cond_record.get('total_prompt_tokens', 0) + cond_record.get('total_completion_tokens', 0))
    stage('case_done', case=case['id'])

# Final A/B comparison summary
ab_table = []
for c in ablation['cases']:
    off = c['conditions'].get('off', {}); on = c['conditions'].get('on', {})
    ab_table.append({
        'case': c['id'], 'difficulty': c['difficulty'],
        'off_passed': off.get('passed'), 'on_passed': on.get('passed'),
        'off_attempts': off.get('n_attempts'), 'on_attempts': on.get('n_attempts'),
        'off_elapsed': round(off.get('elapsed_seconds') or 0, 1),
        'on_elapsed': round(on.get('elapsed_seconds') or 0, 1),
        'off_tokens': (off.get('total_prompt_tokens', 0) + off.get('total_completion_tokens', 0)),
        'on_tokens':  (on.get('total_prompt_tokens', 0) + on.get('total_completion_tokens', 0)),
    })
summary['ab_table'] = ab_table
write_summary()
stage('ablation_done', cases=len(ab_table))
for row in ab_table:
    print(row)
